# Qwen3-8B (như bài báo) trên GPU Kaggle/Colab

Chạy các thí nghiệm bằng Qwen3-8B (Ollama, bản Q8) trên GPU; graph đọc từ `outputs/hierargraph.pkl` trong repo (không dùng Neo4j). Notebook tự nhận ra đang ở Kaggle hay Colab.

Graph, án test, câu hỏi và cache embedding nằm sẵn trong repo; notebook chỉ cần clone, không cần tải dữ liệu lên.

## Kaggle (khuyên dùng: chạy nền, không cần mở trình duyệt)
1. Import notebook này vào Kaggle (Create → Import Notebook).
2. **Settings**: Accelerator = GPU T4 x2 (hoặc P100), Internet = On (cần xác minh số điện thoại tài khoản Kaggle).
3. Chạy thử từng ô một lần (Interactive) tới ô thử Qwen; ổn thì **Save Version → Save & Run All (Commit)**: chạy nền tối đa 12 giờ, có thể tắt máy.
4. Kết quả ở tab **Output** của version: thư mục `outputs/` (kết quả, log từng lượt) và `cache_llm.tar.gz`. Chạy tiếp (khi quá 12 giờ hoặc lỗi): **Add Input** → output của version trước, bỏ các lượt đã xong khỏi `PRESETS`; notebook giải nén cache, các lời gọi đã xong không gọi lại.

## Colab
Runtime = T4 GPU; chạy từng ô (phải giữ tab mở). Cache để trên Drive.

Chọn phần việc của notebook ở biến `NOTEBOOK` (ô đầu tiên). Chấm kết luận hỏi–đáp (`--no-grade` ở đây) làm sau ở máy bằng deepseek, để mọi lượt chạy cùng một người chấm.

In [ ]:
# ===== CHỌN NOTEBOOK NÀY CHẠY PHẦN NÀO =====
# Chạy song song 3 notebook (mỗi notebook 2 GPU): import notebook này 3 lần,
# đặt NOTEBOOK = 'A', 'B', 'C' ở từng bản rồi Save & Run All cả ba.
# Lượt 2 (hỏi–đáp có graph, dài nhất) chia 4 phần cho A và B; C chạy các lượt còn lại.
# Chỉ có 1 notebook: NOTEBOOK = 'TATCA'.
NOTEBOOK = 'A'
PRESETS = {
    #         GPU 0                      GPU 1
    'A':     [[(2, '1/4')],              [(2, '2/4')]],
    'B':     [[(2, '3/4')],              [(2, '4/4')]],
    'C':     [[(1, ''), (3, '')],        [(4, ''), (5, '')]],   # 5 dùng lại cache của 4: cùng GPU
    'TATCA': [[(2, '')],                 [(1, ''), (3, ''), (4, ''), (5, '')]],
}
QUEUES = PRESETS[NOTEBOOK]
print('Notebook', NOTEBOOK, '->', QUEUES)

In [ ]:
import glob, os, shutil, subprocess, sys, time, urllib.request
KAGGLE = os.path.exists('/kaggle')
!nvidia-smi --query-gpu=name,memory.total --format=csv

if KAGGLE:
    WORK = '/kaggle/working'
    OUT = WORK                       # outputs/ và cache_llm/ ở đây thành Output của version
    old_caches = glob.glob('/kaggle/input/**/cache_llm.tar.gz', recursive=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    OUT = '/content/drive/MyDrive/KhoaLuan'
    old_caches = []
# Kaggle: cache ngoài Output (hàng nghìn file nhỏ), gom thành cache_llm.tar.gz sau mỗi lượt
CACHE = '/kaggle/tmp/cache_llm' if KAGGLE else f'{OUT}/cache_llm'
os.makedirs(f'{OUT}/outputs', exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
for old in old_caches:               # Kaggle: tiếp tục từ output của version trước
    subprocess.check_call(['tar', 'xzf', old, '-C', CACHE, '--skip-old-files'])
    print(f'Đã giải nén cache từ {old}')
print('Kaggle' if KAGGLE else 'Colab', '| cache:', len(os.listdir(CACHE)), 'lời gọi')

In [ ]:
BRANCH = 'claude/legalgraphrag-framework-hv23z3'
REPO = f'{WORK}/Agent-Graph-Rag'
!rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/bighousevn/Agent-Graph-Rag.git {REPO}
os.chdir(REPO)
!pip install -q -r requirements.txt
!ls -la outputs/hierargraph.pkl data/processed/cases_vn_test.json data/raw/questions.xlsx

## Biến môi trường

In [ ]:
MODEL = 'qwen3:8b-q8_0'  # Qwen3-8B như bài báo; Q8 vừa một GPU 16 GB cùng ngữ cảnh 16k
os.environ.update({
    'LLM_PROVIDER': 'ollama',
    'LLM_API_KEY': 'ollama',          # Ollama không cần key; biến này chỉ để chọn đúng nhà cung cấp
    'LLM_MODEL': MODEL,
    'LLM_TIMEOUT': '600',
    'CACHE_DIR': CACHE,
})
print('Đã đặt biến môi trường cho', MODEL)

## Ollama + Qwen3-8B

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
# Một server Ollama cho mỗi GPU (Kaggle T4 x2): hai hàng lượt chạy song song.
import torch
N_GPU = max(1, torch.cuda.device_count())
PORTS = [11434 + i for i in range(N_GPU)]
servers = []
for i, port in enumerate(PORTS):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(i), 'OLLAMA_HOST': f'127.0.0.1:{port}',
           'OLLAMA_CONTEXT_LENGTH': '16384', 'OLLAMA_KEEP_ALIVE': '-1'}
    servers.append(subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(f'{OUT}/ollama{i}.log', 'w'), stderr=subprocess.STDOUT))
for port in PORTS:
    for _ in range(60):
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}'); break
        except Exception:
            time.sleep(1)
!OLLAMA_HOST=127.0.0.1:11434 ollama pull {MODEL} 2>&1 | tail -1
print(N_GPU, 'GPU, cổng', PORTS)

In [ ]:
# Thử: trả lời ngay, không suy nghĩ, chạy trên GPU (dừng notebook nếu không đạt).
# Lần gọi đầu nạp model lên GPU (~1 phút); đo thời gian ở lần thứ hai.
sys.path.insert(0, REPO)
from vn_legal_graph.config import AppConfig
from vn_legal_graph.llm import LLMClient
from vn_legal_graph.judge.judge_law import parse_bool
for port in PORTS:
    os.environ['LLM_BASE_URL'] = f'http://127.0.0.1:{port}/v1'
    c = LLMClient(AppConfig.from_env_file('/nonexistent'))
    q = 'Trả lời đúng một từ true hoặc false: trộm cắp tài sản là tội phạm?'
    t = time.time()
    c._ollama_chat(q, 16)
    load = time.time() - t
    t = time.time()
    raw, truncated = c._ollama_chat(q, 16)
    took = time.time() - t
    print(port, repr(raw), f'nạp model {load:.0f}s, trả lời {took:.1f}s')
    assert not truncated and parse_bool(raw) is True, 'Qwen vẫn suy nghĩ hoặc không trả lời true/false'
    assert took < 15, f'Một câu true/false mất {took:.0f}s: có thể đang chạy trên CPU hoặc vẫn suy nghĩ'
!OLLAMA_HOST=127.0.0.1:11434 ollama ps

## Chạy thí nghiệm

| # | lượt | notebook |
|---|---|---|
| 1 | hỏi–đáp, Qwen không graph (187 câu) | C |
| 2 | hỏi–đáp, pipeline graph (187 câu), chia 4 phần | A (phần 1, 2), B (phần 3, 4) |
| 3 | vụ án, Qwen không graph (55 án) | C |
| 4 | vụ án, như bài gốc: Auditor lọc cứng | C |
| 5 | vụ án, bỏ Auditor + Adjudicator xem án tương tự (dùng lại cache của 4) | C |

Kết quả (`outputs/*.json`, ghi sau mỗi câu/án) và cache được lưu mỗi 15 phút, nên nếu bị cắt ở 12 giờ vẫn giữ được phần đã chạy. Cache của lần chạy 2026-10-07 (qua cổng OpenAI, Qwen còn suy nghĩ) không dùng lại.

In [ ]:
# QUEUES chọn ở ô đầu notebook. Một GPU thì gộp các hàng lại.
if N_GPU == 1:
    QUEUES = [[job for q in QUEUES for job in q]]
COMMANDS = {
    1: 'python scripts/run_qa_pilot.py --auto-gold --n 1000 --without-graph --no-grade --tag full',
    2: 'python scripts/run_qa_pilot.py --auto-gold --n 1000 --rerank --no-grade --tag full',
    3: 'python scripts/run_case_pipeline.py --without-graph',
    4: 'python scripts/run_case_pipeline.py --auditor loc --tag loc',
    5: 'python scripts/run_case_pipeline.py --auditor bo-qua --adjudicator-context an --tag boqua_an',
}
import threading

def save():
    for f in glob.glob('outputs/*.json'):
        shutil.copy(f, f'{OUT}/outputs/')
    if KAGGLE:
        subprocess.call(['tar', 'czf', f'{OUT}/cache_llm.tar.gz.tmp', '-C', CACHE, '.'])
        os.replace(f'{OUT}/cache_llm.tar.gz.tmp', f'{OUT}/cache_llm.tar.gz')

def log_of(r, shard):
    return f"{OUT}/outputs/luot{r}{'_phan' + shard.replace('/', '-') if shard else ''}.log"

def run_queue(gpu, runs):
    env = {**os.environ, 'LLM_BASE_URL': f'http://127.0.0.1:{PORTS[gpu]}/v1', 'PYTHONUNBUFFERED': '1'}
    for r, shard in runs:
        t = time.time()
        cmd = COMMANDS[r] + (f' --shard {shard}' if shard else '')
        code_ = subprocess.call(f'{cmd} > {log_of(r, shard)} 2>&1', shell=True, env=env)
        print(f'GPU {gpu} | lượt {r} {shard}: mã thoát {code_}, {(time.time() - t) / 60:.0f} phút', flush=True)

workers = [threading.Thread(target=run_queue, args=(g, q)) for g, q in enumerate(QUEUES)]
for w in workers:
    w.start()
while any(w.is_alive() for w in workers):  # lưu mỗi 15 phút: bị cắt ở 12 giờ vẫn còn kết quả
    for w in workers:
        w.join(timeout=900 / len(workers))
    save()
    done = {f'{r} {s}'.strip(): sum(1 for l in open(log_of(r, s), encoding='utf-8') if l.startswith('['))
            for q in QUEUES for r, s in q if os.path.exists(log_of(r, s))}
    print(time.strftime('%H:%M'), 'đã xong (câu/án):', done, flush=True)
save()
for q in QUEUES:
    for r, s in q:
        lines = open(log_of(r, s), encoding='utf-8').read().splitlines()
        print(f'\n===== Lượt {r} {s}: {COMMANDS[r]}')
        print('\n'.join([l for l in lines if not l.startswith('  ') and not l.startswith('Warning')][-14:]))

## Sau khi chạy xong (ở máy)

Tải `outputs/*.json` của cả 3 notebook về `outputs/` của repo, gộp các phần của lượt 2, rồi chấm kết luận hỏi–đáp bằng deepseek:

```
python scripts/run_qa_pilot.py --merge outputs/qa_pilot_qwen3_8b-q8_0_full_phan*-4.json --merge-out outputs/qa_pilot_qwen3_8b-q8_0_full.json
python scripts/run_qa_pilot.py --grade-only outputs/qa_llm_only_qwen3_8b-q8_0_full.json
python scripts/run_qa_pilot.py --grade-only outputs/qa_pilot_qwen3_8b-q8_0_full.json
```

Điểm vụ án (accuracy, F1) đã có trong file, không cần LLM chấm.